# AdversarialQA preprocessing

Same pipeline as `squad_preprocessing.ipynb` (max length 384, stride 128, seed 42,
BERT / RoBERTa / DeBERTa / DistilBERT), adapted for AdversarialQA:

- `adversarialQA` is the union of `dbidaf`, `dbert` and `droberta`, but its metadata only says
  `Combined`. The combined set is rebuilt from the three sub-configs so each record keeps its
  adversary model.
- The `test` split has no gold answers (empty answer arrays), so it cannot be scored and is not saved.
- All train and validation records are answerable (no SQuAD-style unanswerable questions).
- AdversarialQA is used for robustness evaluation, so only the validation splits are tokenized.
  The cleaned train splits are saved untokenized.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from collections import Counter

In [2]:
PROJECT_ROOT = Path.cwd().parent.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "adversarial_qa"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "adversarial_qa"

CONFIGS = [
    "dbert",
    "dbidaf",
    "adversarialQA",
    "droberta"
]

SPLITS = ["train", "validation", "test"]

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
datasets = {}

for config in CONFIGS:
    datasets[config] = {}

    for split in SPLITS:
        file_path = RAW_DIR / config / split / "0000.parquet"

        df = pd.read_parquet(file_path)

        datasets[config][split] = df

        print(
            f"{config} | {split} | "
            f"Shape: {df.shape}"
        )

dbert | train | Shape: (10000, 6)
dbert | validation | Shape: (1000, 6)
dbert | test | Shape: (1000, 6)
dbidaf | train | Shape: (10000, 6)
dbidaf | validation | Shape: (1000, 6)
dbidaf | test | Shape: (1000, 6)
adversarialQA | train | Shape: (30000, 6)
adversarialQA | validation | Shape: (3000, 6)
adversarialQA | test | Shape: (3000, 6)
droberta | train | Shape: (10000, 6)
droberta | validation | Shape: (1000, 6)
droberta | test | Shape: (1000, 6)


In [4]:
for config in CONFIGS:

    print("\n" + "=" * 60)
    print(f"CONFIGURATION: {config}")
    print("=" * 60)

    df = datasets[config]["train"]

    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes)

    print("\nFirst record:")
    display(df.head(1))


CONFIGURATION: dbert

Column names:
['id', 'title', 'context', 'question', 'answers', 'metadata']

Data types:
id             str
title          str
context        str
question       str
answers     object
metadata    object
dtype: object

First record:


,id,title,context,question,answers,metadata
0,dab017ed8a1c27c6afa2d8618abc3a477a4edffc,Empiricism,"A generation later, the Irish Anglican bishop,...",what concept is mentioned last?,"{'text': ['subjective idealism'], 'answer_star...","{'split': 'train', 'model_in_the_loop': 'BERT-..."



CONFIGURATION: dbidaf

Column names:
['id', 'title', 'context', 'question', 'answers', 'metadata']

Data types:
id             str
title          str
context        str
question       str
answers     object
metadata    object
dtype: object

First record:


,id,title,context,question,answers,metadata
0,821607441c173838196c4d1500c2ab21a044e6b0,Yale_University,Slack (2003) compares three groups that conduc...,what year were the research groups compared,"{'text': ['2003'], 'answer_start': [7]}","{'split': 'train', 'model_in_the_loop': 'BiDAF'}"



CONFIGURATION: adversarialQA

Column names:
['id', 'title', 'context', 'question', 'answers', 'metadata']

Data types:
id             str
title          str
context        str
question       str
answers     object
metadata    object
dtype: object

First record:


,id,title,context,question,answers,metadata
0,7ba1e8f4261d3170fcf42e84a81dd749116fae95,Brain,Another approach to brain function is to exami...,What sare the benifts of the blood brain barrir?,"{'text': ['isolated from the bloodstream'], 'a...","{'split': 'train', 'model_in_the_loop': 'Combi..."



CONFIGURATION: droberta

Column names:
['id', 'title', 'context', 'question', 'answers', 'metadata']

Data types:
id             str
title          str
context        str
question       str
answers     object
metadata    object
dtype: object

First record:


,id,title,context,question,answers,metadata
0,12cf36866b656dc4f254081fe6796ea1be2f6d43,Napoleon,"When he became First Consul and later Emperor,...",What jewelry like accessories did he wear?,"{'text': ['Légion d'honneur star, medal and ri...","{'split': 'train', 'model_in_the_loop': 'RoBER..."


In [5]:
df = datasets["adversarialQA"]["train"]

for column in df.columns:
    print(f"\nColumn: {column}")
    print("Value:", repr(df[column].iloc[0]))


Column: id
Value: '7ba1e8f4261d3170fcf42e84a81dd749116fae95'

Column: title
Value: 'Brain'

Column: context
Value: 'Another approach to brain function is to examine the consequences of damage to specific brain areas. Even though it is protected by the skull and meninges, surrounded by cerebrospinal fluid, and isolated from the bloodstream by the blood–brain barrier, the delicate nature of the brain makes it vulnerable to numerous diseases and several types of damage. In humans, the effects of strokes and other types of brain damage have been a key source of information about brain function. Because there is no ability to experimentally control the nature of the damage, however, this information is often difficult to interpret. In animal studies, most commonly involving rats, it is possible to use electrodes or locally injected chemicals to produce precise patterns of damage and then examine the consequences for behavior.'

Column: question
Value: 'What sare the benifts of the blood br

# 1. Initialize preprocessing configuration

In [6]:
import random
from datetime import datetime

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Tokenization configuration (same as SQuAD 2.0)
MAX_LENGTH = 384
DOC_STRIDE = 128

MODEL_CHECKPOINTS = {
    "bert": "bert-base-uncased",
    "roberta": "roberta-base",
    "deberta": "microsoft/deberta-v3-base",
    "distilbert": "distilbert-base-uncased"
}

# Sub-configs with a single adversary model in the loop
SUB_CONFIGS = ["dbidaf", "dbert", "droberta"]

# Splits with gold answers
LABELLED_SPLITS = ["train", "validation"]

print("Project root:", PROJECT_ROOT)
print("Raw path exists:", RAW_DIR.exists())
print("Processed path:", PROCESSED_DIR)
print("Max sequence length:", MAX_LENGTH)
print("Document stride:", DOC_STRIDE)
print("Random seed:", RANDOM_SEED)
print("Models:", list(MODEL_CHECKPOINTS.keys()))

Project root: /Users/vaibhav/Documents/Projects/Domain-Adaptive-and-Robustness-Aware-Question-Answering-
Raw path exists: True
Processed path: /Users/vaibhav/Documents/Projects/Domain-Adaptive-and-Robustness-Aware-Question-Answering-/data/processed/adversarial_qa
Max sequence length: 384
Document stride: 128
Random seed: 42
Models: ['bert', 'roberta', 'deberta', 'distilbert']


# 2. Verify the combined config against the sub-configs

In [7]:
# adversarialQA should be exactly the union of dbidaf, dbert and droberta
for split in SPLITS:
    combined_ids = set(datasets["adversarialQA"][split]["id"])
    sub_ids = set().union(
        *[set(datasets[c][split]["id"]) for c in SUB_CONFIGS]
    )

    print(
        f"{split}: combined={len(combined_ids)} | "
        f"sub-configs union={len(sub_ids)} | "
        f"identical={combined_ids == sub_ids}"
    )

print("\nadversarialQA model_in_the_loop values:")
print(
    datasets["adversarialQA"]["train"]["metadata"]
    .apply(lambda m: m["model_in_the_loop"])
    .value_counts()
)

train: combined=29965 | sub-configs union=29965 | identical=True
validation: combined=3000 | sub-configs union=3000 | identical=True
test: combined=3000 | sub-configs union=3000 | identical=True

adversarialQA model_in_the_loop values:
metadata
Combined    30000
Name: count, dtype: int64


# 3. Check the test split has no gold answers

In [8]:
for config in CONFIGS:
    test_df = datasets[config]["test"]
    n_with_answers = test_df["answers"].apply(
        lambda a: len(a["text"]) > 0
    ).sum()

    print(
        f"{config} test: {len(test_df)} records | "
        f"records with gold answers: {n_with_answers}"
    )

print("\nThe test split is not scoreable and is excluded from the processed data.")

dbert test: 1000 records | records with gold answers: 0
dbidaf test: 1000 records | records with gold answers: 0
adversarialQA test: 3000 records | records with gold answers: 0
droberta test: 1000 records | records with gold answers: 0

The test split is not scoreable and is excluded from the processed data.


# 4. Standardize answer annotations

In [9]:
def standardize_answers(df, config):

    df = df.copy()

    # Flatten metadata
    df["adversary"] = config
    df["model_in_the_loop"] = df["metadata"].apply(
        lambda m: m["model_in_the_loop"]
    )

    # Extract all answer annotations
    df["answer_texts"] = df["answers"].apply(
        lambda x: list(x["text"]) if x is not None else []
    )

    df["answer_starts"] = df["answers"].apply(
        lambda x: [int(s) for s in x["answer_start"]] if x is not None else []
    )

    # Determine answerability
    df["is_impossible"] = df["answer_texts"].apply(
        lambda x: len(x) == 0
    )

    # Primary answer and its character offset
    df["answer_text"] = df["answer_texts"].apply(
        lambda x: x[0] if x else ""
    )

    df["answer_start"] = df["answer_starts"].apply(
        lambda x: x[0] if x else -1
    )

    # Keep the nested answers column (SQuAD format) for tokenization
    df["answers"] = df.apply(
        lambda r: {"text": r["answer_texts"], "answer_start": r["answer_starts"]},
        axis=1
    )

    return df.drop(columns=["metadata"])


processed = {
    config: {
        split: standardize_answers(datasets[config][split], config)
        for split in LABELLED_SPLITS
    }
    for config in SUB_CONFIGS
}

for config in SUB_CONFIGS:
    for split in LABELLED_SPLITS:
        df = processed[config][split]
        print(
            f"{config} | {split} | shape={df.shape} | "
            f"answerable={(~df['is_impossible']).sum()} | "
            f"unanswerable={df['is_impossible'].sum()} | "
            f"model_in_the_loop={df['model_in_the_loop'].unique().tolist()}"
        )

print("\nColumns:")
print(processed["dbidaf"]["train"].columns.tolist())

display(
    processed["dbert"]["train"][
        ["question", "answer_text", "answer_start", "is_impossible", "adversary"]
    ].head(3)
)

dbidaf | train | shape=(10000, 12) | answerable=10000 | unanswerable=0 | model_in_the_loop=['BiDAF']
dbidaf | validation | shape=(1000, 12) | answerable=1000 | unanswerable=0 | model_in_the_loop=['BiDAF']
dbert | train | shape=(10000, 12) | answerable=10000 | unanswerable=0 | model_in_the_loop=['BERT-Large']
dbert | validation | shape=(1000, 12) | answerable=1000 | unanswerable=0 | model_in_the_loop=['BERT-Large']
droberta | train | shape=(10000, 12) | answerable=10000 | unanswerable=0 | model_in_the_loop=['RoBERTa-Large']
droberta | validation | shape=(1000, 12) | answerable=1000 | unanswerable=0 | model_in_the_loop=['RoBERTa-Large']

Columns:
['id', 'title', 'context', 'question', 'answers', 'adversary', 'model_in_the_loop', 'answer_texts', 'answer_starts', 'is_impossible', 'answer_text', 'answer_start']


,question,answer_text,answer_start,is_impossible,adversary
0,what concept is mentioned last?,subjective idealism,742,False,dbert
1,what concept is mentioned first?,atheism,159,False,dbert
2,what group is mentioned last?,humans,614,False,dbert


# 5. Validate answer character offsets

In [10]:
def validate_answer_offsets(df, name):

    total = valid = 0
    invalid_examples = []

    for _, row in df.iterrows():
        context = row["context"]

        for answer, start in zip(row["answer_texts"], row["answer_starts"]):
            total += 1
            extracted = context[start:start + len(answer)]

            if extracted == answer:
                valid += 1
            else:
                invalid_examples.append({
                    "id": row["id"],
                    "answer": answer,
                    "answer_start": start,
                    "extracted_text": extracted
                })

    print(
        f"{name}: annotations={total} | valid={valid} | "
        f"invalid={total - valid} | "
        f"accuracy={round(valid / total * 100, 2) if total else 0} %"
    )

    return pd.DataFrame(invalid_examples)


invalid_offsets = {}

for config in SUB_CONFIGS:
    for split in LABELLED_SPLITS:
        invalid_offsets[(config, split)] = validate_answer_offsets(
            processed[config][split], f"{config}/{split}"
        )

total_invalid = sum(len(v) for v in invalid_offsets.values())
print("\nTotal invalid offsets:", total_invalid)

dbidaf/train: annotations=10000 | valid=10000 | invalid=0 | accuracy=100.0 %
dbidaf/validation: annotations=1000 | valid=1000 | invalid=0 | accuracy=100.0 %


dbert/train: annotations=10000 | valid=10000 | invalid=0 | accuracy=100.0 %
dbert/validation: annotations=1000 | valid=1000 | invalid=0 | accuracy=100.0 %


droberta/train: annotations=10000 | valid=10000 | invalid=0 | accuracy=100.0 %
droberta/validation: annotations=1000 | valid=1000 | invalid=0 | accuracy=100.0 %

Total invalid offsets: 0


# 6. Check empty fields and answer lengths

In [11]:
for config in SUB_CONFIGS:
    for split in LABELLED_SPLITS:
        df = processed[config][split]

        empty_questions = (df["question"].str.strip() == "").sum()
        empty_contexts = (df["context"].str.strip() == "").sum()
        empty_answers = (df["answer_text"].str.strip() == "").sum()
        answer_words = df["answer_text"].str.split().str.len()

        print(
            f"{config}/{split}: empty questions={empty_questions} | "
            f"empty contexts={empty_contexts} | "
            f"empty answers={empty_answers} | "
            f"answer words mean={answer_words.mean():.2f}, max={answer_words.max()}"
        )

dbidaf/train: empty questions=0 | empty contexts=0 | empty answers=0 | answer words mean=3.80, max=138
dbidaf/validation: empty questions=0 | empty contexts=0 | empty answers=0 | answer words mean=2.94, max=26
dbert/train: empty questions=0 | empty contexts=0 | empty answers=0 | answer words mean=3.78, max=282
dbert/validation: empty questions=0 | empty contexts=0 | empty answers=0 | answer words mean=2.96, max=34
droberta/train: empty questions=0 | empty contexts=0 | empty answers=0 | answer words mean=4.34, max=150
droberta/validation: empty questions=0 | empty contexts=0 | empty answers=0 | answer words mean=3.17, max=49


# 7. Investigate duplicate records and annotation conflicts

In [12]:
def answer_signature(row):
    return (
        tuple(row["answer_texts"]),
        tuple(int(x) for x in row["answer_starts"])
    )


def investigate_duplicates(df, name):
    data = df.copy()
    data["answer_signature"] = data.apply(answer_signature, axis=1)

    conflicts = (
        data.groupby(["question", "context"])["answer_signature"].nunique()
    )
    conflicts = conflicts[conflicts > 1]

    print(f"\n{'=' * 45}")
    print(name)
    print(f"{'=' * 45}")
    print("Duplicate IDs:", data["id"].duplicated().sum())
    print(
        "Duplicate question-context pairs:",
        data.duplicated(subset=["question", "context"]).sum()
    )
    print(
        "Exact duplicate records:",
        data.duplicated(
            subset=["id", "title", "context", "question", "answer_signature"]
        ).sum()
    )
    print("Annotation conflict groups:", len(conflicts))

    if len(conflicts) > 0:
        display(
            data[
                data.set_index(["question", "context"]).index.isin(conflicts.index)
            ][["question", "answer_texts", "answer_starts"]].head(5)
        )


for config in SUB_CONFIGS:
    for split in LABELLED_SPLITS:
        investigate_duplicates(processed[config][split], f"{config}/{split}")


dbidaf/train


Duplicate IDs: 0
Duplicate question-context pairs: 0
Exact duplicate records: 0
Annotation conflict groups: 0

dbidaf/validation
Duplicate IDs: 0
Duplicate question-context pairs: 0
Exact duplicate records: 0
Annotation conflict groups: 0

dbert/train
Duplicate IDs: 0
Duplicate question-context pairs: 0
Exact duplicate records: 0
Annotation conflict groups: 0

dbert/validation
Duplicate IDs: 0
Duplicate question-context pairs: 0
Exact duplicate records: 0
Annotation conflict groups: 0



droberta/train
Duplicate IDs: 0
Duplicate question-context pairs: 0
Exact duplicate records: 0
Annotation conflict groups: 0

droberta/validation
Duplicate IDs: 0
Duplicate question-context pairs: 0
Exact duplicate records: 0
Annotation conflict groups: 0


# 8. Check leakage across splits and adversaries

In [13]:
# Train vs validation within each adversary
for config in SUB_CONFIGS:
    train_df = processed[config]["train"]
    val_df = processed[config]["validation"]

    print(
        f"{config}: ID overlap={len(set(train_df['id']) & set(val_df['id']))} | "
        f"question overlap={len(set(train_df['question']) & set(val_df['question']))} | "
        f"context overlap={len(set(train_df['context']) & set(val_df['context']))}"
    )

# Combined train vs combined validation
all_train = pd.concat([processed[c]["train"] for c in SUB_CONFIGS])
all_val = pd.concat([processed[c]["validation"] for c in SUB_CONFIGS])

print(
    f"\ncombined: ID overlap={len(set(all_train['id']) & set(all_val['id']))} | "
    f"question overlap={len(set(all_train['question']) & set(all_val['question']))} | "
    f"context overlap={len(set(all_train['context']) & set(all_val['context']))}"
)

# Overlap of AdversarialQA contexts with SQuAD 2.0 (AdversarialQA reuses SQuAD passages)
squad_raw = PROJECT_ROOT / "data" / "raw" / "squad_v2"
squad_contexts = set()
for split in ["train", "validation"]:
    squad_contexts |= set(pd.read_parquet(squad_raw / split / "0000.parquet")["context"])

for split, df in [("train", all_train), ("validation", all_val)]:
    contexts = set(df["context"])
    print(
        f"AdversarialQA {split} contexts also in SQuAD 2.0: "
        f"{len(contexts & squad_contexts)} / {len(contexts)}"
    )

dbidaf: ID overlap=0 | question overlap=1 | context overlap=0
dbert: ID overlap=0 | question overlap=12 | context overlap=0
droberta: ID overlap=0 | question overlap=15 | context overlap=0

combined: ID overlap=0 | question overlap=35 | context overlap=0


AdversarialQA train contexts also in SQuAD 2.0: 2627 / 2648
AdversarialQA validation contexts also in SQuAD 2.0: 160 / 416


# 9. Build the combined set with adversary labels

In [14]:
combined = {
    split: (
        pd.concat([processed[c][split] for c in SUB_CONFIGS])
        .reset_index(drop=True)
    )
    for split in LABELLED_SPLITS
}

for split in LABELLED_SPLITS:
    # Same IDs as the downloaded adversarialQA config
    same_ids = set(combined[split]["id"]) == set(datasets["adversarialQA"][split]["id"])
    print(f"combined {split}: {combined[split].shape} | matches adversarialQA IDs: {same_ids}")
    print(combined[split]["adversary"].value_counts().to_dict())

processed["combined"] = combined
OUTPUT_CONFIGS = SUB_CONFIGS + ["combined"]

combined train: (30000, 12) | matches adversarialQA IDs: True
{'dbidaf': 10000, 'dbert': 10000, 'droberta': 10000}
combined validation: (3000, 12) | matches adversarialQA IDs: True
{'dbidaf': 1000, 'dbert': 1000, 'droberta': 1000}


# 10. Save processed datasets and preprocessing metadata in data/processed/

In [15]:
SAVE_COLUMNS = [
    "id", "title", "context", "question", "answers",
    "answer_texts", "answer_starts", "is_impossible",
    "answer_text", "answer_start", "adversary", "model_in_the_loop"
]

saved_sizes = {}

for config in OUTPUT_CONFIGS:
    out_dir = PROCESSED_DIR / config
    out_dir.mkdir(parents=True, exist_ok=True)

    for split in LABELLED_SPLITS:
        df = processed[config][split][SAVE_COLUMNS]
        path = out_dir / f"{split}.parquet"
        df.to_parquet(path, index=False)

        saved_sizes[f"{config}/{split}"] = len(df)
        print(f"Saved {config}/{split}: {df.shape}")

metadata = {
    "dataset": "AdversarialQA",
    "role": "adversarial_evaluation",
    "random_seed": RANDOM_SEED,
    "max_length": MAX_LENGTH,
    "doc_stride": DOC_STRIDE,
    "models": MODEL_CHECKPOINTS,
    "configs": {
        "dbidaf": "adversary: BiDAF",
        "dbert": "adversary: BERT-Large",
        "droberta": "adversary: RoBERTa-Large",
        "combined": "dbidaf + dbert + droberta (same IDs as the adversarialQA config)"
    },
    "splits": saved_sizes,
    "excluded": {
        "test": "no gold answers in any test record; not scoreable"
    },
    "tokenized_splits": ["validation"],
    "created_at": datetime.now().isoformat()
}

with open(PROCESSED_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("\nMetadata saved to", PROCESSED_DIR / "metadata.json")

Saved dbidaf/train: (10000, 12)
Saved dbidaf/validation: (1000, 12)
Saved dbert/train: (10000, 12)
Saved dbert/validation: (1000, 12)
Saved droberta/train: (10000, 12)
Saved droberta/validation: (1000, 12)
Saved combined/train: (30000, 12)
Saved combined/validation: (3000, 12)

Metadata saved to /Users/vaibhav/Documents/Projects/Domain-Adaptive-and-Robustness-Aware-Question-Answering-/data/processed/adversarial_qa/metadata.json


# 11. Tokenize validation splits for BERT, RoBERTa, DeBERTa and DistilBERT

In [16]:
%pip install sentencepiece protobuf


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: /Users/vaibhav/Documents/Projects/Domain-Adaptive-and-Robustness-Aware-Question-Answering-/.venv/bin/python3.14 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [17]:
from transformers import AutoTokenizer
from datasets import load_dataset

TOKENIZED_DIR = PROCESSED_DIR / "tokenized"
TOKENIZED_DIR.mkdir(parents=True, exist_ok=True)

eval_datasets = {
    config: load_dataset(
        "parquet",
        data_files=str(PROCESSED_DIR / config / "validation.parquet"),
        split="train"
    )
    for config in OUTPUT_CONFIGS
}


# Same tokenization and answer alignment as SQuAD 2.0
def preprocess_function(examples, tokenizer):

    questions = [q.strip() for q in examples["question"]]

    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        truncation="only_second",
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    raw_offsets = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []
    example_ids = []
    saved_offsets = []
    is_impossible = []

    for i, offsets in enumerate(raw_offsets):

        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        example_ids.append(str(examples["id"][sample_index]))

        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = tokenized.sequence_ids(i)

        context_positions = [
            j for j, sid in enumerate(sequence_ids) if sid == 1
        ]

        # Keep offsets only for context tokens
        saved_offsets.append([
            list(offsets[j]) if sequence_ids[j] == 1 else [-1, -1]
            for j in range(len(offsets))
        ])

        # Unanswerable question
        if not answers["answer_start"] or not answers["text"]:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            is_impossible.append(True)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        if not context_positions:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            is_impossible.append(False)
            continue

        start_token = next(
            (j for j in context_positions
             if offsets[j][0] <= start_char < offsets[j][1]),
            None
        )

        end_token = next(
            (j for j in reversed(context_positions)
             if offsets[j][0] < end_char <= offsets[j][1]),
            None
        )

        # Answer not fully present in this window
        if start_token is None or end_token is None or start_token > end_token:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            start_positions.append(start_token)
            end_positions.append(end_token)

        is_impossible.append(False)

    tokenized["example_id"] = example_ids
    tokenized["offset_mapping"] = saved_offsets
    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    tokenized["is_impossible"] = is_impossible

    return tokenized


feature_counts = {}

for model_name, checkpoint in MODEL_CHECKPOINTS.items():

    print(f"\nProcessing: {model_name.upper()}")

    tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)

    if tokenizer.cls_token_id is None:
        raise ValueError(f"{model_name} has no CLS token ID")

    for config, dataset in eval_datasets.items():

        features = dataset.map(
            lambda examples: preprocess_function(examples, tokenizer),
            batched=True,
            remove_columns=dataset.column_names
        )

        # Structural validation
        invalid = too_long = 0
        for row in features:
            ids = row["input_ids"]
            if len(ids) > MAX_LENGTH:
                too_long += 1
            if not (0 <= row["start_positions"] <= row["end_positions"] < len(ids)):
                invalid += 1

        if invalid or too_long:
            raise ValueError(f"Validation failed: {model_name}/{config}")

        output_path = TOKENIZED_DIR / model_name / f"{config}_validation.parquet"
        output_path.parent.mkdir(parents=True, exist_ok=True)
        features.to_parquet(str(output_path))

        feature_counts[f"{model_name}/{config}"] = features.num_rows

        print(
            f"{config}_validation: {features.num_rows} features | "
            f"Invalid positions: {invalid} | Overlength sequences: {too_long}"
        )

print("\nTokenization completed.")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]


Processing: BERT


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

... (43 outputs left out)

droberta_validation: 1003 features | Invalid positions: 0 | Overlength sequences: 0


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

combined_validation: 3010 features | Invalid positions: 0 | Overlength sequences: 0

Tokenization completed.


# 12. Final preprocessing validation: gold answer-span round trip

Each labelled feature's token span is mapped back to characters and compared with the gold answer.
RoBERTa's byte-level BPE sometimes merges the last answer character with the punctuation after it
(for example `%)`, `",` or `(−`), so the span can start or end one punctuation mark wide. These are counted as
*token-boundary* cases rather than failures: no token boundary exists at the gold edge, and EM/F1
normalization strips punctuation. The same effect appears in the SQuAD 2.0 RoBERTa features.

In [18]:
total_passed = 0
total_failed = 0
total_cls = 0
total_boundary = 0

import string
PUNCTUATION = set(string.punctuation)


def is_token_boundary_case(reconstructed, gold_texts):
    # Gold answer starts or ends inside a merged token, e.g. RoBERTa BPE "%)", "\"," or "(−":
    # the span only adds punctuation, which SQuAD-style EM/F1 normalization removes
    rec = reconstructed.strip()
    for g in gold_texts:
        g = g.strip()
        pos = rec.find(g)
        if g and pos >= 0:
            extra = rec[:pos] + rec[pos + len(g):]
            if extra and set(extra) <= PUNCTUATION:
                return True
    return False


for model_name, checkpoint in MODEL_CHECKPOINTS.items():

    print(f"\n{'=' * 60}")
    print(f"MODEL: {model_name.upper()}")
    print("=" * 60)

    tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)
    cls_id = tokenizer.cls_token_id

    for config in OUTPUT_CONFIGS:

        original_df = pd.read_parquet(PROCESSED_DIR / config / "validation.parquet")
        original_df["id"] = original_df["id"].astype(str)
        lookup = original_df.set_index("id").to_dict("index")

        tokenized_df = pd.read_parquet(
            TOKENIZED_DIR / model_name / f"{config}_validation.parquet"
        )

        passed = failed = checked = cls_only = boundary = 0

        for _, feature in tokenized_df.iterrows():

            example_id = str(feature["example_id"])
            if example_id not in lookup:
                failed += 1
                continue

            original = lookup[example_id]
            start = int(feature["start_positions"])
            end = int(feature["end_positions"])
            input_ids = feature["input_ids"]

            # CLS-labelled feature: answer not in this window
            if input_ids[start] == cls_id and input_ids[end] == cls_id:
                cls_only += 1
                continue

            offsets = feature["offset_mapping"]
            start_char = offsets[start][0]
            end_char = offsets[end][1]

            checked += 1

            if start_char < 0 or end_char <= start_char:
                failed += 1
                continue

            reconstructed = original["context"][start_char:end_char]

            # Compare with whitespace stripped: RoBERTa/DeBERTa offsets can
            # include a leading space, and some gold answers end in a space
            matches = any(
                reconstructed.strip() == text.strip()
                for text in original["answer_texts"]
            )

            if matches:
                passed += 1
            elif is_token_boundary_case(reconstructed, original["answer_texts"]):
                boundary += 1
            else:
                failed += 1

        total_passed += passed
        total_failed += failed
        total_cls += cls_only
        total_boundary += boundary

        # Every example must have at least one feature that contains its answer
        answered_ids = set(
            tokenized_df.loc[
                tokenized_df["start_positions"] != tokenized_df["input_ids"].apply(
                    lambda ids: list(ids).index(cls_id)
                ),
                "example_id"
            ].astype(str)
        )
        missing = len(set(lookup) - answered_ids)

        print(
            f"{config}_validation: Checked={checked} | Passed={passed} | "
            f"Boundary={boundary} | Failed={failed} | CLS windows={cls_only} | "
            f"Examples with no answer window={missing}"
        )

print("\n" + "=" * 60)
print("FINAL GOLD ANSWER-SPAN VERIFICATION")
print("=" * 60)
print("Total passed:", total_passed)
print("Token-boundary cases (extra punctuation only):", total_boundary)
print("Total failed:", total_failed)
print("CLS-labelled windows (answer outside window):", total_cls)

if total_failed == 0:
    print("RESULT: ALL CHECKED ANSWER SPANS PASSED")
else:
    print("RESULT: INVESTIGATION REQUIRED")


MODEL: BERT


dbidaf_validation: Checked=1001 | Passed=1001 | Boundary=0 | Failed=0 | CLS windows=2 | Examples with no answer window=0
dbert_validation: Checked=1000 | Passed=1000 | Boundary=0 | Failed=0 | CLS windows=4 | Examples with no answer window=0
droberta_validation: Checked=1000 | Passed=1000 | Boundary=0 | Failed=0 | CLS windows=3 | Examples with no answer window=0


combined_validation: Checked=3001 | Passed=3001 | Boundary=0 | Failed=0 | CLS windows=9 | Examples with no answer window=0

MODEL: ROBERTA


dbidaf_validation: Checked=1001 | Passed=994 | Boundary=7 | Failed=0 | CLS windows=4 | Examples with no answer window=0
dbert_validation: Checked=1000 | Passed=990 | Boundary=10 | Failed=0 | CLS windows=4 | Examples with no answer window=0
droberta_validation: Checked=1000 | Passed=994 | Boundary=6 | Failed=0 | CLS windows=3 | Examples with no answer window=0


combined_validation: Checked=3001 | Passed=2978 | Boundary=23 | Failed=0 | CLS windows=11 | Examples with no answer window=0

MODEL: DEBERTA


dbidaf_validation: Checked=1000 | Passed=1000 | Boundary=0 | Failed=0 | CLS windows=2 | Examples with no answer window=0
dbert_validation: Checked=1000 | Passed=1000 | Boundary=0 | Failed=0 | CLS windows=4 | Examples with no answer window=0
droberta_validation: Checked=1000 | Passed=1000 | Boundary=0 | Failed=0 | CLS windows=3 | Examples with no answer window=0


combined_validation: Checked=3000 | Passed=3000 | Boundary=0 | Failed=0 | CLS windows=9 | Examples with no answer window=0

MODEL: DISTILBERT


dbidaf_validation: Checked=1001 | Passed=1001 | Boundary=0 | Failed=0 | CLS windows=2 | Examples with no answer window=0
dbert_validation: Checked=1000 | Passed=1000 | Boundary=0 | Failed=0 | CLS windows=4 | Examples with no answer window=0
droberta_validation: Checked=1000 | Passed=1000 | Boundary=0 | Failed=0 | CLS windows=3 | Examples with no answer window=0


combined_validation: Checked=3001 | Passed=3001 | Boundary=0 | Failed=0 | CLS windows=9 | Examples with no answer window=0

FINAL GOLD ANSWER-SPAN VERIFICATION
Total passed: 23960
Token-boundary cases (extra punctuation only): 46
Total failed: 0
CLS-labelled windows (answer outside window): 76
RESULT: ALL CHECKED ANSWER SPANS PASSED
